# Chinese Rap Lyrics NER Pipeline (Colab)

端到端流水线：数据清洗 → NER 实体识别 → Bag-of-Entities → K-Means 聚类

## 使用前准备

在你的 Google Drive **根目录**建一个 `rap-data` 文件夹，放入：
```
我的云端硬盘/
  rap-data/
    lyrics_chunks_enriched.csv
    entity_review.csv          （你审核过的，没有就不放）
```

然后从下面第一个 cell 开始依次运行即可。

In [ ]:
# ===== Step 0: Clone repo & install deps =====
!rm -rf /content/rap-lycis-II
!git clone https://github.com/Mo119m/rap-lycis-II.git /content/rap-lycis-II
%cd /content/rap-lycis-II
!git checkout claude/refactor-data-pipeline-i67JA

!pip install -q spacy scikit-learn scipy
!python -m spacy download zh_core_web_trf

In [ ]:
# ===== Step 0.5: Mount Google Drive & copy data =====
import shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = Path('/content/drive/MyDrive/rap-data')

shutil.copy(DRIVE_DATA / 'lyrics_chunks_enriched.csv', 'lyrics_chunks_enriched.csv')
print('[OK] Copied lyrics_chunks_enriched.csv')

review_src = DRIVE_DATA / 'entity_review.csv'
if review_src.exists():
    shutil.copy(review_src, 'configs/entity_review.csv')
    print('[OK] Copied entity_review.csv')
else:
    print('[INFO] No entity_review.csv in Drive, will generate on first run')

print('\n=== Data ready! ===')

In [ ]:
# ===== Init =====
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

OUTPUTS = Path('outputs')
OUTPUTS.mkdir(exist_ok=True)

## 1. 数据加载与清洗

In [ ]:
ARTIST_LYRICS_PATH = OUTPUTS / 'artist_lyrics.csv'

if ARTIST_LYRICS_PATH.exists():
    print(f'[CACHE] Loading from {ARTIST_LYRICS_PATH}')
    artist_lyrics = pd.read_csv(ARTIST_LYRICS_PATH)
    print(f'Loaded: {len(artist_lyrics)} artists')
else:
    from src.data_cleaning import load_and_clean, combine_by_artist

    df_raw = load_and_clean('lyrics_chunks_enriched.csv')
    artist_lyrics = combine_by_artist(df_raw)
    artist_lyrics.to_csv(ARTIST_LYRICS_PATH, index=False)
    print(f'\n[SAVED] {ARTIST_LYRICS_PATH}')

In [ ]:
# 浏览数据
print(f'共 {len(artist_lyrics)} 位艺人')

artist_lyrics['text_len'] = artist_lyrics['combined_text'].str.len()
print(f'\n歌词长度分布:')
print(artist_lyrics['text_len'].describe())

top10 = artist_lyrics.nlargest(10, 'text_len')[['artist', 'text_len']]
print(f'\n歌词最长的 10 位艺人:')
for _, r in top10.iterrows():
    print(f'  {r["artist"]}: {r["text_len"]:,} 字符')

## 2. NER 实体识别

这一步最耗时。跑完后结果自动缓存。

In [ ]:
from src.ner import build_nlp, extract_entities, normalize_entities

SPACY_MODEL = 'zh_core_web_trf'

ENTITIES_PATH = OUTPUTS / 'entities_long.csv'
ENTITIES_RAW_PATH = OUTPUTS / 'entities_raw.csv'

if ENTITIES_PATH.exists():
    print(f'[CACHE] Loading from {ENTITIES_PATH}')
    entity_df = pd.read_csv(ENTITIES_PATH)
    print(f'Loaded: {len(entity_df)} entity mentions')
else:
    import gc

    nlp = build_nlp('configs/rap_lexicon_seed.jsonl', model_name=SPACY_MODEL)
    entity_df = extract_entities(artist_lyrics, nlp, min_entity_len=2)

    del nlp
    gc.collect()
    print('[INFO] Released spaCy model from memory')

    entity_df.to_csv(ENTITIES_RAW_PATH, index=False)

    entity_df = normalize_entities(entity_df)
    entity_df.to_csv(ENTITIES_PATH, index=False)
    print(f'\n[SAVED] {ENTITIES_PATH}')

In [ ]:
from src.ner import entity_summary

print('=== Top 30 高频实体 ===')
top30 = entity_summary(entity_df, top_n=30)
print(top30.to_string(index=False))

In [ ]:
print('=== 标签分布 ===')
label_dist = entity_df['label'].value_counts()
print(label_dist)

print('\n=== 各标签类型 Top-5 实体 ===')
for label in label_dist.index:
    subset = entity_df[entity_df['label'] == label]
    top5 = subset['entity'].value_counts().head(5)
    print(f'\n[{label}] ({len(subset)} mentions)')
    for ent, cnt in top5.items():
        print(f'  {ent}: {cnt}')

## 2.5 实体人工审核与修正

In [ ]:
from src.ner import generate_entity_review, apply_entity_corrections

REVIEW_PATH = 'configs/entity_review.csv'
MIN_COUNT = 2

if not Path(REVIEW_PATH).exists():
    GLOBAL_MIN_COUNT = 10
    generate_entity_review(entity_df, output_path=REVIEW_PATH, top_n=100,
                           global_min_count=GLOBAL_MIN_COUNT)
    print('\n>>> 请打开 configs/entity_review.csv 填写 action 列，然后重新 Run 这个 cell <<<')
else:
    n_before = len(entity_df)
    entity_df = apply_entity_corrections(entity_df, review_path=REVIEW_PATH, min_count=MIN_COUNT)
    if len(entity_df) != n_before:
        entity_df.to_csv(ENTITIES_PATH, index=False)
        print(f'[SAVED] Updated {ENTITIES_PATH}')
        for f in OUTPUTS.glob('bag_of_entities.*'):
            f.unlink()
        for f in [OUTPUTS / 'artist_clusters.csv', OUTPUTS / 'cluster_entity_summary.csv']:
            if f.exists():
                f.unlink()
        print('[INFO] Cleared clustering cache')

## 3. Bag-of-Entities 矩阵与聚类

In [ ]:
from src.clustering import EntityMatrix, build_bag_of_entities, run_kmeans, summarize_clusters
import gc

BOE_PATH = OUTPUTS / 'bag_of_entities'
CLUSTERS_PATH = OUTPUTS / 'artist_clusters.csv'
SUMMARY_PATH = OUTPUTS / 'cluster_entity_summary.csv'

N_CLUSTERS = 6
MIN_ENTITY_FREQ = 3

if (OUTPUTS / 'bag_of_entities.npz').exists() and CLUSTERS_PATH.exists() and SUMMARY_PATH.exists():
    print('[CACHE] Loading clustering results')
    entity_matrix = EntityMatrix.load_npz(str(BOE_PATH))
    assignments = pd.read_csv(CLUSTERS_PATH)
    summaries = pd.read_csv(SUMMARY_PATH)
    print(f'Loaded: {entity_matrix.shape[0]} artists x {entity_matrix.shape[1]} entities, '
          f'{assignments["cluster"].nunique()} clusters')
else:
    try:
        del artist_lyrics
    except NameError:
        pass
    gc.collect()

    entity_matrix = build_bag_of_entities(entity_df, min_entity_freq=MIN_ENTITY_FREQ)
    nnz = entity_matrix.data.nnz
    total = entity_matrix.shape[0] * entity_matrix.shape[1]
    print(f'Bag-of-Entities: {entity_matrix.shape[0]} artists x {entity_matrix.shape[1]} entities')
    print(f'Non-zero: {nnz} / {total} ({nnz/total*100:.1f}% dense)')

    del entity_df
    gc.collect()

    assignments, centers = run_kmeans(entity_matrix, n_clusters=N_CLUSTERS, random_state=42)
    summaries = summarize_clusters(centers, entity_matrix.entities, top_k=15)
    del centers
    gc.collect()

    entity_matrix.save_npz(str(BOE_PATH))
    assignments.to_csv(CLUSTERS_PATH, index=False)
    summaries.to_csv(SUMMARY_PATH, index=False)
    print(f'\n[SAVED] bag_of_entities.npz, {CLUSTERS_PATH.name}, {SUMMARY_PATH.name}')

    entity_df = pd.read_csv(ENTITIES_PATH)

In [ ]:
# 各聚类的艺人
print(f'=== 聚类结果（K={assignments["cluster"].nunique()}）===')
for cluster_id in sorted(assignments['cluster'].unique()):
    artists_in_cluster = assignments[assignments['cluster'] == cluster_id]['artist'].tolist()
    print(f'\n--- Cluster {cluster_id} ({len(artists_in_cluster)} artists) ---')
    print(', '.join(artists_in_cluster))

In [ ]:
# 各聚类代表实体
print('=== 各聚类代表实体（Top 15）===')
for cluster_name in summaries['cluster'].unique():
    cluster_data = summaries[summaries['cluster'] == cluster_name]
    cluster_data = cluster_data[cluster_data['centroid_weight'] > 0]
    print(f'\n--- {cluster_name} ---')
    for _, r in cluster_data.iterrows():
        print(f'  {r["entity"]}: {r["centroid_weight"]:.4f}')

## 4. 质量检查

In [ ]:
if 'artist_lyrics' not in dir() or not isinstance(artist_lyrics, pd.DataFrame):
    artist_lyrics = pd.read_csv(ARTIST_LYRICS_PATH)

CHECK_ARTIST = artist_lyrics.iloc[0]['artist']
print(f'=== 质量检查: {CHECK_ARTIST} ===')

artist_entities = entity_df[entity_df['artist'] == CHECK_ARTIST]
print(f'该艺人共提取 {len(artist_entities)} 个实体 mention')
print(f'\n实体频率:')
for (ent, label), cnt in artist_entities.groupby(['entity', 'label']).size().sort_values(ascending=False).head(20).items():
    print(f'  {ent} ({label}): {cnt}')

text_sample = artist_lyrics[artist_lyrics['artist'] == CHECK_ARTIST]['combined_text'].iloc[0]
print(f'\n歌词片段（前 500 字）:')
print(text_sample[:500])

In [ ]:
print('=== 可疑实体（可能是噪声）===')

entity_counts = entity_df['entity'].value_counts()
singletons = entity_counts[entity_counts == 1]
print(f'\n只出现 1 次的实体数: {len(singletons)} / {len(entity_counts)} ({len(singletons)/len(entity_counts)*100:.1f}%)')
print('样本:', singletons.head(20).index.tolist())

long_entities = entity_df[entity_df['entity'].str.len() > 10]['entity'].unique()
print(f'\n长度 > 10 的实体 ({len(long_entities)} 个):')
for e in long_entities[:20]:
    print(f'  "{e}"')